# 06 K 近邻 KNN

KNN 是一种非常直观的模型：预测一个新样本时，先找训练集中离它最近的 K 个样本，再用这些邻居投票或取平均。


## 0. 学习目标和阅读地图

KNN 的直觉最简单，但坑也很典型。你需要掌握：

1. 为什么 KNN 几乎没有训练过程。
2. 距离度量和特征缩放为什么决定模型表现。
3. K 值如何控制过拟合和欠拟合。
4. 为什么高维数据里 KNN 经常变差。


## 1. 数学逻辑

常用欧氏距离：

$$d(x,z)=\sqrt{\sum_j(x_j-z_j)^2}$$

分类时，取距离最近的 K 个邻居：

$$N_K(x)=\text{K nearest samples to }x$$

再做多数投票：

$$\hat y=\text{mode}\{y_i|x_i\in N_K(x)\}$$

KNN 几乎没有显式训练，主要计算发生在预测阶段。


## 1.1 推导拆开看：KNN 在估计局部标签分布

KNN 分类可以看成在新点附近估计类别概率：

$$P(y=c|x) \approx \frac{1}{K}\sum_{i\in N_K(x)} I(y_i=c)$$

然后选概率最大的类别。

当 `K=1` 时，边界非常灵活，训练误差可能很低，但对噪声敏感。当 `K` 很大时，局部信息被平均掉，边界更平滑，但可能欠拟合。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.datasets import make_moons
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
X, y = make_moons(n_samples=250, noise=0.25, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)


## 1.2 数据流和标准化

KNN 完全依赖距离。如果一个特征的数值范围是 0 到 10000，另一个是 0 到 1，欧氏距离几乎只会被第一个特征支配。

所以代码使用 `StandardScaler`，把每个特征转成均值约 0、标准差约 1。


In [ ]:
# 从零实现：KNN 分类

def knn_predict_one(x, X_train, y_train, k=5):
    distances = np.sqrt(np.sum((X_train - x) ** 2, axis=1))
    neighbor_ids = np.argsort(distances)[:k]
    votes = Counter(y_train[neighbor_ids])
    return votes.most_common(1)[0][0]

def knn_predict(X_new, X_train, y_train, k=5):
    return np.array([knn_predict_one(x, X_train, y_train, k) for x in X_new])

for k in [1, 5, 21]:
    pred = knn_predict(X_test_s, X_train_s, y_train, k=k)
    print(f'k={k:2d} | accuracy={accuracy_score(y_test, pred):.3f}')


## 1.3 从零实现代码怎么读

`knn_predict_one` 做了三件事：

1. 计算测试点到所有训练点的距离。
2. 取距离最小的 K 个训练样本。
3. 对这些邻居的标签做投票。

这也解释了 KNN 的缺点：训练便宜，但预测时要和很多训练样本算距离。


In [ ]:
# sklearn 实战：KNeighborsClassifier
model = KNeighborsClassifier(n_neighbors=7)
model.fit(X_train_s, y_train)
pred = model.predict(X_test_s)
print('sklearn accuracy:', round(accuracy_score(y_test, pred), 3))

xx, yy = np.meshgrid(np.linspace(X_train_s[:,0].min()-1, X_train_s[:,0].max()+1, 180),
                     np.linspace(X_train_s[:,1].min()-1, X_train_s[:,1].max()+1, 180))
grid = np.c_[xx.ravel(), yy.ravel()]
zz = model.predict(grid).reshape(xx.shape)
plt.contourf(xx, yy, zz, alpha=0.25, cmap='coolwarm')
plt.scatter(X_train_s[:,0], X_train_s[:,1], c=y_train, cmap='coolwarm', edgecolor='k', s=25)
plt.title('KNN 决策边界')
plt.show()


In [ ]:
# 诊断：K 值扫描，观察过拟合/欠拟合趋势
ks = list(range(1, 32, 2))
train_acc, test_acc = [], []
for k in ks:
    m_knn = KNeighborsClassifier(n_neighbors=k)
    m_knn.fit(X_train_s, y_train)
    train_acc.append(accuracy_score(y_train, m_knn.predict(X_train_s)))
    test_acc.append(accuracy_score(y_test, m_knn.predict(X_test_s)))

plt.plot(ks, train_acc, marker='o', label='train')
plt.plot(ks, test_acc, marker='o', label='test')
plt.title('K 值对 KNN 表现的影响')
plt.xlabel('K')
plt.ylabel('accuracy')
plt.legend()
plt.show()


## 2.1 如何诊断 KNN

如果训练准确率很高、测试准确率低，K 可能太小。如果两者都低，K 可能太大，或者特征本身不能区分类别。

KNN 的诊断重点不是 loss，而是距离是否有意义：特征缩放、异常值、高维稀疏性都会影响邻居关系。


## 2. 常见误区

- KNN 对特征尺度极其敏感，通常必须标准化。
- K 太小容易过拟合，K 太大容易欠拟合。
- 高维空间中距离会变得不可靠，这叫维度灾难。

## 3. 小实验

- 改 `k`，观察边界平滑程度。
- 去掉标准化，观察效果变化。
- 把 `noise` 调大，看 KNN 何时变得不稳定。


## 5. 复习清单

- KNN 是基于实例的学习，预测时才做主要计算。
- K 小更灵活，K 大更平滑。
- 特征缩放对 KNN 非常关键。
- 高维空间里距离会变得越来越不可靠。
